# JavaScript in the Browser

A practical guide to the DOM, event handling, and HTML form interaction using modern ECMAScript.

## Table of Contents

- [Module 01: Document Object Model Fundamentals](#module-01)
- [Module 02: Working with the DOM](#module-02)
- [Module 03: Event Handling Fundamentals](#module-03)
- [Module 04: Mouse and Keyboard Event Handling](#module-04)
- [Module 05: HTML Form Handling](#module-05)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

def render_diagram(image_path, width=900):
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="{width}"/></div>'))

---
## Module 01: Document Object Model Fundamentals <a id="module-01"></a>

The Document Object Model (DOM) is a programming interface for HTML documents. It represents the page as a tree of nodes that JavaScript can read and manipulate.

### What is the DOM?

The DOM is the browser's in-memory representation of an HTML document as a tree of **nodes**. When the browser parses an HTML file it constructs this tree, and JavaScript can then query, modify, add, or remove any node in the tree at runtime.

The left panel below shows the raw HTML source; the right panel shows the corresponding DOM tree the browser builds from it.

In [ ]:
render_diagram(Path("assets/01-dom-html-to-tree.png"))

### DOM Version History

The DOM specification has evolved through several levels under the W3C, and is now maintained as a continuously updated **Living Standard** by WHATWG.

In [ ]:
render_diagram(Path("assets/02-dom-version-history-timeline.png"))

| Year | Specification |
|------|--------------|
| 1998 | DOM Level 1 — basic document structure and traversal |
| 2000 | DOM Level 2 — events, CSS styling, namespaces |
| 2004 | DOM Level 3 — load/save, XPath, keyboard events |
| 2004– | DOM Living Standard — continuously updated, the current authority |

> In practice you target the **DOM Living Standard**. DOM Levels are historical milestones, not distinct targets.

### Demo: Accessing a DOM Element

See [`01-dom-fundamentals-demos/`](01-dom-fundamentals-demos/) for the Globoticket demo application used throughout this module.

---
## Module 02: Working with the DOM <a id="module-02"></a>

This module covers how to select, read, modify, add, and remove DOM elements using JavaScript.

### DOM Structure

Every HTML document maps to a tree. Each HTML tag becomes an **element node**; text content and attributes become child nodes.

In [ ]:
render_diagram(Path("assets/03-dom-structure-tree.png"))

#### Terminology

- **Node** — any object in the tree (element, text, attribute, comment)
- **Element node** — represents an HTML tag
- **Attribute** — a named property on an element node (e.g. `href="..."`)
- **Property** — a JavaScript property on the DOM object that may or may not correspond 1:1 to an HTML attribute

### Selecting DOM Elements

Five primary APIs for locating elements in the tree:

In [ ]:
render_diagram(Path("assets/04-dom-selection-methods.png"))

| API | Returns | Notes |
|-----|---------|-------|
| `getElementById(id)` | Single element | Fastest; id must be unique |
| `getElementsByTagName(tag)` | Live HTMLCollection | All elements with that tag |
| `getElementsByClassName(cls)` | Live HTMLCollection | All elements with that class |
| `querySelector(selector)` | First matching element | Full CSS selector support |
| `querySelectorAll(selector)` | Static NodeList | All matching elements |

Prefer `querySelector`/`querySelectorAll` for new code — they accept any CSS selector and return predictable static collections.

### DOM Navigation

Once you have a reference to a node, you can traverse the tree using navigation properties:

In [ ]:
render_diagram(Path("assets/05-dom-navigation-properties.png"))

| Property | Description |
|----------|-------------|
| `parentNode` | The parent node |
| `childNodes` | All child nodes (NodeList) |
| `firstChild` / `lastChild` | First or last child node |
| `nextSibling` / `previousSibling` | Adjacent sibling nodes |

> These properties include **text nodes** (whitespace). Use `firstElementChild`, `nextElementSibling`, etc. to skip text nodes and work only with element nodes.

### Reading and Modifying DOM Content

```javascript
let el = document.getElementById("...");

el.innerHTML = "...";           // Set or get HTML content (parses tags)
el.innerText = "...";           // Set or get visible text
el.textContent = "...";         // Set or get all text (including hidden)
el.firstChild.nodeValue = "..."; // Set text node value directly
el.setAttribute("name", "value"); // Set any HTML attribute
```

### Manipulating CSS

Three approaches to change an element's styles at runtime:

- **`style` attribute** — `el.setAttribute("style", "color: orange")` — replaces the entire inline style string
- **`style` property** — `el.style.color = "orange"` — sets individual CSS properties as JS object properties
- **`classList` property** — `el.classList.add("my-orange")` — adds/removes/toggles CSS class names (preferred)

`classList` is the cleanest approach: it keeps styles in CSS files and avoids clobbering other inline styles.

### Removing DOM Elements

In [ ]:
render_diagram(Path("assets/06-removing-dom-elements.png"))

Use `node.remove()` for all new code — the legacy `parentNode.removeChild(node)` pattern exists only in older codebases.

### Adding DOM Elements

To create new content and insert it into the page:

In [ ]:
render_diagram(Path("assets/07-adding-dom-elements.png"))

```javascript
// Shorthand: write raw HTML into a placeholder element
let el = document.getElementById("placeholder");
el.innerHTML = "<span>some text</span>";

// Explicit node creation (safer — avoids XSS risk from untrusted strings)
let new_p      = document.createElement("p");
let new_p_text = document.createTextNode("Hello");
new_p.appendChild(new_p_text);   // text node → new element
el.appendChild(new_p);           // new element → page
```

> Prefer `createElement` + `textContent` over `innerHTML` when inserting user-supplied data, to avoid XSS vulnerabilities.

### Demo: Adding DOM Elements

See [`02-working-with-dom-demos/`](02-working-with-dom-demos/) for worked examples of element creation and manipulation.

---
## Module 03: Event Handling Fundamentals <a id="module-03"></a>

Events are notifications fired by the browser when something happens — a user click, a key press, the DOM finishing loading. JavaScript registers **listeners** that run in response.

### Event APIs — History

Three event-binding APIs exist. Only the W3C standard (`addEventListener`) should be used in new code:

In [ ]:
render_diagram(Path("assets/08-event-apis.png"))

| API | Origin | Status |
|-----|--------|--------|
| `el.onclick = fn` | Netscape | Avoid — only one listener per event |
| `el.attachEvent()` | Internet Explorer | Obsolete — IE only |
| `el.addEventListener()` | W3C Standard | **Use this** — multiple listeners, full control |

### Adding an Event Listener

```javascript
let el = document.getElementById("some-id");

el.addEventListener("click", (e) => {
    // event handling code
});
```

To register a listener that fires **only once** and then automatically removes itself:

```javascript
el.addEventListener("click", (e) => {
    // runs once, then removed
}, { once: true });
```

### Event Arguments

Every listener receives an event object `e` with at minimum:

- **`e.type`** — the event type string (e.g. `"click"`, `"keydown"`)
- **`e.target`** — the element that originally triggered the event
- **Event-specific properties** — coordinates for mouse events, key info for keyboard events, etc.

### Event Bubbling

When an event fires on an element, it **bubbles upward** through the DOM tree — each ancestor also receives the event in turn, bottom to top.

In [ ]:
render_diagram(Path("assets/09-event-bubbling.png"))

Bubbling enables **event delegation**: attach one listener to a parent element to handle events from many children, instead of attaching a listener to each child individually.

To stop bubbling: `e.stopPropagation()`.

### Event Capturing

Capturing is the opposite of bubbling — the event travels **top to bottom** from the document root to the target. It fires before bubbling begins.

Enable capturing by passing `true` (or `{ capture: true }`) as the third argument:

In [ ]:
render_diagram(Path("assets/10-event-capturing.png"))

```javascript
el.addEventListener("click", handler, { capture: true });
```

**Phase order for a single click:** capturing (root → target) → target phase → bubbling (target → root).

### Preventing Default Behavior

Many events have a browser default action (e.g. a link navigates, a form submits). Call `e.preventDefault()` to suppress it:

```javascript
el.addEventListener("click", (e) => {
    e.preventDefault(); // stop link navigation
    // custom logic
});
```

### Waiting Until the DOM Is Ready

Scripts in `<head>` run before the DOM is fully parsed. Use `DOMContentLoaded` to defer code until the DOM is ready:

```javascript
document.addEventListener("DOMContentLoaded", () => {
    // safe to query DOM here
});
```

Or place `<script>` tags at the end of `<body>` — the DOM is then fully parsed before the script runs.

### Demo: Preventing Default Behavior / Waiting for DOM Ready

See [`03-event-handling-demos/`](03-event-handling-demos/) for examples of `preventDefault()` and `DOMContentLoaded`.

---
## Module 04: Mouse and Keyboard Event Handling <a id="module-04"></a>

This module covers the specific events fired by mouse and keyboard interactions, and the data they carry.

### Mouse Events — Hover

Three events track the mouse moving relative to an element's boundary:

In [ ]:
render_diagram(Path("assets/11-mouse-events-hover.png"))

- **`mouseenter`** — fires once when the pointer enters the element (does not bubble)
- **`mousemove`** — fires continuously as the pointer moves within the element
- **`mouseleave`** — fires once when the pointer exits the element (does not bubble)

### Mouse Events — Click Sequence

A full click on an element fires three events in order: `mousedown` → `mouseup` → `click`.

### Mouse Event Data

Every mouse event object carries coordinate properties describing where the pointer was:

| Property | Coordinate origin |
|----------|-------------------|
| `clientX` / `clientY` | Viewport (visible browser window) |
| `pageX` / `pageY` | Full page including scrolled-out area |
| `screenX` / `screenY` | Physical screen |
| `offsetX` / `offsetY` | Target element's own top-left corner |
| `button` / `buttons` | Which mouse button(s) are pressed |
| `ctrlKey` (and `shiftKey`, `altKey`, `metaKey`) | Whether modifier keys are held |

### Demo: Drag and Drop

See [`04-mouse-keyboard-demos/`](04-mouse-keyboard-demos/) for a drag-and-drop implementation using mouse events.

### Keyboard Events

A complete key press fires three events in sequence:

In [ ]:
render_diagram(Path("assets/12-keyboard-events-flow.png"))

- **`keydown`** — fires when the key is pressed down; repeats if held
- **`keypress`** — deprecated; fires for printable characters only; use `keydown` instead
- **`keyup`** — fires when the key is released

The dashed loop back indicates that `keydown` repeats continuously while the key is held before `keyup` fires.

### Keyboard Event Data — `code` vs `key`

The critical distinction: **`code`** identifies the physical key position on the keyboard (layout-independent); **`key`** gives the character produced by that key (layout-dependent).

The next two diagrams show the same physical keys circled on two different keyboard layouts. The `code` value is the same on both (`KeyZ`, `KeyY`); the `key` value differs (`z`/`y` on a US layout vs `y`/`z` on a German QWERTZ layout).

In [ ]:
render_diagram(Path("assets/13-keyboard-layout-mac.png"))
render_diagram(Path("assets/14-keyboard-layout-windows.png"))

### Keyboard Event Data Properties

In [ ]:
render_diagram(Path("assets/15-keyboard-event-data.png"))

| Property | Description | Status |
|----------|-------------|--------|
| `code` | Physical key identifier (e.g. `"KeyZ"`, `"ArrowUp"`) | **Use this** for shortcuts |
| `key` | Character or key name produced (e.g. `"z"`, `"ArrowUp"`) | **Use this** for text input |
| `keyCode` | Numeric key code | **Deprecated** — avoid |
| `charCode` | Numeric character code | **Deprecated** — avoid |

Use `code` when you want to bind keyboard shortcuts regardless of the user's keyboard language. Use `key` when you care about the character the user typed.

---
## Module 05: HTML Form Handling <a id="module-05"></a>

HTML forms are the primary mechanism for collecting user input in a web page. JavaScript provides APIs to access form elements, read their values, and control form submission.

### Accessing HTML Form Elements

Forms and their elements can be accessed either through standard DOM selection methods or through the specialized `document.forms` collection:

1. **DOM selection methods** — `querySelector`, `getElementById`, etc.
2. **Index access** — `document.forms[0].elements[2]`
3. **Name access** — `document.forms["my-form"].elements["my-element"]`

### Accessing HTML Form Element Values

Different form controls expose their current value through different properties:

In [ ]:
render_diagram(Path("assets/16-form-element-values.png"))

| Control type | Property to read |
|---|---|
| `<input type="text">`, `<textarea>` | `.value` |
| `<select>` | `.options`, `.selectedIndex`, `.value` |
| `<input type="checkbox">`, `<input type="radio">` | `.checked`, `.value` |

### Preventing Form Submission

By default a form submits to the server when the user clicks the submit button, causing a page reload. JavaScript can intercept this:

In [ ]:
render_diagram(Path("assets/17-form-submission-prevention.png"))

Three patterns, from oldest to modern:

```html
<!-- 1. Always prevent (HTML attribute) -->
<form onsubmit="return false">

<!-- 2. Prevent only if validation fails (HTML attribute) -->
<form onsubmit="return checkForm(this)">
```

```javascript
// 3. Modern — prevent in an event listener (preferred)
form.onsubmit = (e) => {
    // validate here
    e.preventDefault(); // stop submission
};
```

Use the `addEventListener` + `e.preventDefault()` pattern for new code — it keeps behaviour in JavaScript, supports multiple listeners, and is fully compatible with the event model.

### Demo: Form Validation with JavaScript

See [`05-form-handling-demos/`](05-form-handling-demos/) for a complete form validation example including value reading and submission prevention.